In [1]:
pip install pandas numpy scikit-learn imbalanced-learn xgboost matplotlib shap

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --force-reinstall "numpy==1.26.4" "pandas"

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached pandas-3.1.1-cp312-cp312-win_amd64.whl.metadata (16 kB)
INFO: pip is looking at multiple versions of pandas to determine which version is compatible with other requirements. This could take a while.
  Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl.metadata (18 kB)
  Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached python_dateutil-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached tzdata-2026.6-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)
Using cached pandas-3.0.6-cp312-cp312-win_amd64.whl (9.7 MB)
Using cached python_dateutil-4.0.0-py3-none-any.whl (92 kB)
Using cached tzdata-2026.6-py2.py3-none-any.whl (348 kB)
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2026.6
    Uninstalling tzdata-2026.6:
      Successfully uninstalled tzdata-2026.6
  Attempting uninstall: numpy
    Foun

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
botocore 1.34.69 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 4.0.0 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.54.1 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
streamlit 1.32.0 requires pandas<3,>=1.3.0, but you have pandas 3.0.6 which is incompatible.
streamlit 1.32.0 requires rich<14,>=10.14.0, but you have rich 14.3.3 which is incompatible.


In [3]:
import os
import time
import warnings
 
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate, RandomizedSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, precision_recall_curve,
                             matthews_corrcoef)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier

C:\Users\AYUSHMAAN\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\AYUSHMAAN\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [6]:
DATA_PATH = "creditcard.csv"
OUT = "results_extended"
SEED = 42
RUN_EDA = True
RUN_CV = True
RUN_RESAMPLING = True
RUN_THRESHOLD = True
RUN_TUNING = True
RUN_SHAP = True
 
os.makedirs(OUT, exist_ok=True)
df = pd.read_csv(DATA_PATH)
X = df.drop(columns="Class")
y = df["Class"]
SPW = (y == 0).sum() / (y == 1).sum()
FEATURES = ["Time", "Amount"] + [c for c in X.columns if c not in ("Time", "Amount")]

In [10]:
def prep():
    return ColumnTransformer([("sc", StandardScaler(), ["Time", "Amount"])],
                             remainder="passthrough")
 
 
def make_model(name, weighted):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=1000, random_state=SEED,
                                  class_weight="balanced" if weighted else None)
    if name == "Random Forest":
        return RandomForestClassifier(
            n_estimators=100, max_depth=12, n_jobs=-1, random_state=SEED,
            class_weight="balanced_subsample" if weighted else None)
    return XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                         tree_method="hist", n_jobs=-1, random_state=SEED,
                         eval_metric="logloss",
                         scale_pos_weight=SPW if weighted else 1)
 
 
def build(name, strategy, sampler=None):
    steps = [("prep", prep())]
    if strategy == "SMOTE":
        steps.append(("smote", SMOTE(random_state=SEED)))
    if sampler is not None:
        steps.append(("sampler", sampler))
    steps.append(("clf", make_model(name, strategy == "Class weighting")))
    return Pipeline(steps)
 
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
 
 
def evaluate(y_true, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {"Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred),
            "F1": f1_score(y_true, pred),
            "MCC": matthews_corrcoef(y_true, pred),
            "ROC_AUC": roc_auc_score(y_true, proba),
            "PR_AUC": average_precision_score(y_true, proba),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn}
 

In [20]:
# ---------------------------------------------------------
# 1. EXPLORATORY DATA ANALYSIS (EDA)
# ---------------------------------------------------------

if RUN_EDA:
    print("Running EDA...")

    import os

    os.makedirs(OUT, exist_ok=True)

    # 1. Transaction amount statistics by class
    stats = df.groupby("Class")["Amount"].describe()

    stats.round(3).to_csv(
        f"{OUT}/eda_amount_stats.csv"
    )

    # 2. Transaction amount and time distributions
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))

    # Logarithmic bins for transaction amounts
    bins = np.logspace(
        -2,
        np.log10(df["Amount"].max() + 1),
        40
    )

    for c, lab, col in [
        (0, "Legitimate", "#4c72b0"),
        (1, "Fraud", "#c44e52")
    ]:
        ax[0].hist(
            df.loc[df["Class"] == c, "Amount"] + 0.01,
            bins=bins,
            alpha=0.6,
            density=True,
            label=lab,
            color=col
        )

    ax[0].set_xscale("log")
    ax[0].set_xlabel("Transaction Amount (log scale)")
    ax[0].set_ylabel("Density")
    ax[0].set_title("Transaction Amount by Class")
    ax[0].legend()

    # Transaction time distribution
    for c, lab, col in [
        (0, "Legitimate", "#4c72b0"),
        (1, "Fraud", "#c44e52")
    ]:
        ax[1].hist(
            df.loc[df["Class"] == c, "Time"] / 3600,
            bins=48,
            alpha=0.6,
            density=True,
            label=lab,
            color=col
        )

    ax[1].set_xlabel("Hours Since First Transaction")
    ax[1].set_ylabel("Density")
    ax[1].set_title("Transaction Time by Class")
    ax[1].legend()

    plt.tight_layout()

    plt.savefig(
        f"{OUT}/eda_amount_time.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # 3. Correlation analysis
    numeric_df = df.select_dtypes(include=np.number)

    corr = (
        numeric_df.corr()["Class"]
        .drop("Class")
        .sort_values()
    )

    # Select 8 lowest and 8 highest correlations
    top = pd.concat([
        corr.head(8),
        corr.tail(8)
    ])

    plt.figure(figsize=(7, 5))

    top.plot(
        kind="barh",
        color=[
            "#c44e52" if v < 0 else "#4c72b0"
            for v in top
        ]
    )

    plt.xlabel("Pearson Correlation with Class")
    plt.title("Features Most Correlated with Fraud")
    plt.tight_layout()

    plt.savefig(
        f"{OUT}/eda_correlation.png",
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # 4. Save correlation values
    corr.to_csv(
        f"{OUT}/eda_correlations.csv",
        header=["Correlation"]
    )

    print("EDA completed successfully!")
    print(f"Results saved in: {OUT}")

Running EDA...
EDA completed successfully!
Results saved in: results_extended


In [22]:
# ---------------------------------------------------------------- 2. CV
if RUN_CV:
    print("5-fold cross-validation (this is the slowest part) ...")
    skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
    scoring = {"precision": "precision", "recall": "recall", "f1": "f1",
               "roc_auc": "roc_auc", "pr_auc": "average_precision"}
    rows = []
    for strat in ["Baseline", "Class weighting", "SMOTE"]:
        for name in ["Logistic Regression", "Random Forest", "XGBoost"]:
            t0 = time.time()
            cv = cross_validate(build(name, strat), X, y, cv=skf,
                                scoring=scoring, n_jobs=1)
            row = {"Model": name, "Strategy": strat}
            for k in scoring:
                v = cv[f"test_{k}"]
                row[f"{k}_mean"] = v.mean()
                row[f"{k}_std"] = v.std()
            row["time_s"] = time.time() - t0
            rows.append(row)
            print(f"  {name:20s} {strat:16s} F1={row['f1_mean']:.3f}"
                  f"+/-{row['f1_std']:.3f}  PR-AUC={row['pr_auc_mean']:.3f}")
            pd.DataFrame(rows).round(4).to_csv(f"{OUT}/cv_results.csv", index=False)

5-fold cross-validation (this is the slowest part) ...
  Logistic Regression  Baseline         F1=0.726+/-0.030  PR-AUC=0.758
  Random Forest        Baseline         F1=0.856+/-0.020  PR-AUC=0.847
  XGBoost              Baseline         F1=0.856+/-0.035  PR-AUC=0.848
  Logistic Regression  Class weighting  F1=0.121+/-0.006  PR-AUC=0.742
  Random Forest        Class weighting  F1=0.835+/-0.022  PR-AUC=0.833
  XGBoost              Class weighting  F1=0.865+/-0.024  PR-AUC=0.859


C:\Users\AYUSHMAAN\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\AYUSHMAAN\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid

  Logistic Regression  SMOTE            F1=0.111+/-0.006  PR-AUC=0.740
  Random Forest        SMOTE            F1=0.703+/-0.019  PR-AUC=0.830
  XGBoost              SMOTE            F1=0.734+/-0.033  PR-AUC=0.852


In [23]:
## ------------------------------------------------- 3. Other resampling
if RUN_RESAMPLING:
    print("Other resampling methods with XGBoost ...")
    samplers = {
        "None": None,
        "Random undersampling (1:10)": RandomUnderSampler(sampling_strategy=0.1,
                                                          random_state=SEED),
        "Random undersampling (1:1)": RandomUnderSampler(random_state=SEED),
        "SMOTE": SMOTE(random_state=SEED),
        "Borderline-SMOTE": BorderlineSMOTE(random_state=SEED),
        "ADASYN": ADASYN(random_state=SEED),
    }
    rows = []
    for lab, smp in samplers.items():
        steps = [("prep", prep())]
        if smp is not None:
            steps.append(("sampler", smp))
        steps.append(("clf", make_model("XGBoost", False)))
        pipe = Pipeline(steps)
        t0 = time.time()
        pipe.fit(X_train, y_train)
        dt = time.time() - t0
        r = evaluate(y_test, pipe.predict_proba(X_test)[:, 1])
        r.update({"Sampler": lab, "Train_time_s": dt})
        rows.append(r)
        print(f"  {lab:30s} F1={r['F1']:.3f} PR-AUC={r['PR_AUC']:.3f}")
    pd.DataFrame(rows).round(4).to_csv(f"{OUT}/resampling_methods.csv", index=False)

Other resampling methods with XGBoost ...
  None                           F1=0.857 PR-AUC=0.844
  Random undersampling (1:10)    F1=0.601 PR-AUC=0.752
  Random undersampling (1:1)     F1=0.066 PR-AUC=0.655
  SMOTE                          F1=0.718 PR-AUC=0.856
  Borderline-SMOTE               F1=0.827 PR-AUC=0.863
  ADASYN                         F1=0.680 PR-AUC=0.838


In [28]:
# ------------------------------------------------ 4. Threshold tuning
if RUN_THRESHOLD:
    print("Threshold tuning ...")
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=0.25, stratify=y_train, random_state=SEED)
    configs = [("Random Forest", "Baseline"), ("XGBoost", "Class weighting"),
               ("XGBoost", "Baseline")]
    rows, sweep = [], []
    plt.figure(figsize=(6, 3.8))
    for name, strat in configs:
        pipe = build(name, strat)
        pipe.fit(X_tr, y_tr)
        p_val = pipe.predict_proba(X_val)[:, 1]
        p_te = pipe.predict_proba(X_test)[:, 1]
        prec, rec, thr = precision_recall_curve(y_val, p_val)
        f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
        best_thr = float(thr[np.argmax(f1s)])
        for t, tag in [(0.5, "default 0.5"), (best_thr, "tuned on validation")]:
            r = evaluate(y_test, p_te, t)
            r.update({"Model": name, "Strategy": strat, "Threshold": round(t, 4),
                      "Rule": tag})
            rows.append(r)
        for t in np.arange(0.05, 1.0, 0.05):
            r = evaluate(y_test, p_te, t)
            r.update({"Model": name, "Strategy": strat, "Threshold": round(t, 2)})
            sweep.append(r)
        s = pd.DataFrame([x for x in sweep if x["Model"] == name
                          and x["Strategy"] == strat])
        plt.plot(s["Threshold"], s["F1"], label=f"{name} ({strat})")
    plt.xlabel("Decision threshold")
    plt.ylabel("Test F1")
    plt.title("F1 versus decision threshold")
    plt.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{OUT}/threshold_f1_curve.png", dpi=200)
    plt.close()
    pd.DataFrame(rows).round(4).to_csv(f"{OUT}/threshold_tuning.csv", index=False)
    pd.DataFrame(sweep).round(4).to_csv(f"{OUT}/threshold_sweep.csv", index=False)
 

Threshold tuning ...


In [29]:
# ------------------------------------------- 5. Hyperparameter tuning
if RUN_TUNING:
    print("Tuning XGBoost with RandomizedSearchCV ...")
    pipe = Pipeline([("prep", prep()),
                     ("clf", make_model("XGBoost", True))])
    space = {"clf__n_estimators": [100, 200, 300, 400],
             "clf__max_depth": [3, 4, 5, 6, 8],
             "clf__learning_rate": [0.03, 0.05, 0.1, 0.2],
             "clf__subsample": [0.7, 0.85, 1.0],
             "clf__colsample_bytree": [0.6, 0.8, 1.0],
             "clf__min_child_weight": [1, 3, 5]}
    search = RandomizedSearchCV(
        pipe, space, n_iter=15, scoring="average_precision",
        cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
        random_state=SEED, n_jobs=1, refit=True, verbose=1)
    t0 = time.time()
    search.fit(X_train, y_train)
    r = evaluate(y_test, search.predict_proba(X_test)[:, 1])
    r.update({"Search_time_s": time.time() - t0,
              "Best_CV_PR_AUC": search.best_score_,
              "Best_params": str(search.best_params_)})
    pd.DataFrame([r]).round(4).to_csv(f"{OUT}/tuning_result.csv", index=False)
    print("  best params:", search.best_params_)
 

Tuning XGBoost with RandomizedSearchCV ...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
  best params: {'clf__subsample': 0.7, 'clf__n_estimators': 300, 'clf__min_child_weight': 1, 'clf__max_depth': 8, 'clf__learning_rate': 0.05, 'clf__colsample_bytree': 0.6}


In [31]:
# ------------------------------------------------------------ 6. SHAP
if RUN_SHAP:
    try:
        import shap
        print("SHAP ...")
        pre = prep()
        Xt = pd.DataFrame(pre.fit_transform(X_train), columns=FEATURES)
        Xe = pd.DataFrame(pre.transform(X_test), columns=FEATURES)
        model = make_model("XGBoost", True)
        model.fit(Xt, y_train)
        sample = Xe.sample(2000, random_state=SEED)
        # make sure some fraud cases are in the sample
        fraud_idx = np.where(y_test.values == 1)[0]
        sample = pd.concat([sample, Xe.iloc[fraud_idx]]).drop_duplicates()
        sv = shap.TreeExplainer(model).shap_values(sample)
        plt.figure()
        shap.summary_plot(sv, sample, plot_type="bar", max_display=12, show=False)
        plt.tight_layout()
        plt.savefig(f"{OUT}/shap_bar.png", dpi=200, bbox_inches="tight")
        plt.close()
        plt.figure()
        shap.summary_plot(sv, sample, max_display=12, show=False)
        plt.tight_layout()
        plt.savefig(f"{OUT}/shap_beeswarm.png", dpi=200, bbox_inches="tight")
        plt.close()
    except ImportError:
        print("shap not installed, skipping (pip install shap)")
 
print(f"\nDone. Files are in ./{OUT}/")

SHAP ...

Done. Files are in ./results_extended/
